In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# نسخ ملفات الـ Zip من الدرايف لمساحة عمل كولاب
!cp "/content/drive/MyDrive/dataset_hybrid_4to1.zip" "/content/"
!cp "/content/drive/MyDrive/valid.zip" "/content/"

# فك الضغط في فولدرات منفصلة (حرف q عشان ميطبعش أسماء كل الصور ويزحم الشاشة)
!unzip -q /content/dataset_hybrid_4to1.zip -d /content/dataset_hybrid_4to1
!unzip -q /content/valid.zip -d /content/valid

print("✅ Data Unzipped Successfully!")

✅ Data Unzipped Successfully!


In [ ]:
!pip install ultralytics -q
import ultralytics
ultralytics.checks()

Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 45.3/112.6 GB disk)


In [ ]:
import yaml

# تحديث المسارات لتتطابق مع الفولدرات المتداخلة (Nested Folders)
data = {
    'train': '/content/dataset_hybrid_4to1/dataset_hybrid_4to1/images', # تم تعديل المسار
    'val': '/content/valid/valid/images',                               # تم تعديل المسار
    'nc': 2,
    'names': ['Crack', 'Pothole']
}

# كتابة الملف
with open('/content/data.yaml', 'w') as f:
    yaml.dump(data, f, default_flow_style=False)

print("✅ data.yaml updated successfully with correct ALL paths!")

✅ data.yaml updated successfully with correct ALL paths!


In [ ]:
from ultralytics import YOLO
import os

# =========================================================
# RoadEye - JapanV2 Final Pilot Run (20 Epochs)
# Fine-tuning from previous best.pt
# Goal:
# Improve pothole detection safely before full 80-epoch run
# =========================================================

# Load previous best model (Fine-tuning)
model = YOLO('/content/drive/MyDrive/best.pt')

# Google Drive output folder
project_path = '/content/drive/MyDrive/JapanV2'
os.makedirs(project_path, exist_ok=True)

print("🚀 Starting JapanV2 PILOT RUN (20 Epochs)...")
print("📦 Saving all weights/results directly to Google Drive...")
print("🎯 Strategy: Fine-tuning on balanced 4:1 dataset")
print("🧠 Focus: Improve pothole recall before full training")

# =========================================================
# PILOT TRAINING CONFIGURATION
# =========================================================

results = model.train(
    # Dataset YAML path
    data='/content/data.yaml',

    # -----------------------------------------------------
    # Core Training (Start with 1024 first)
    # -----------------------------------------------------
    epochs=20,
    imgsz=1024,             # Keep same resolution as previous best.pt
    batch=6,                # Safe starting point for T4 GPU
    workers=2,

    # -----------------------------------------------------
    # Save directly to Google Drive
    # -----------------------------------------------------
    project=project_path,
    name='pilot_diagnostic_4to1',

    # -----------------------------------------------------
    # Fine-tuning Settings (Use same optimizer family)
    # -----------------------------------------------------
    optimizer='SGD',
    lr0=0.003,              # Lower than original training
    lrf=0.1,
    momentum=0.937,
    weight_decay=0.0005,

    # Warmup
    warmup_epochs=2.0,
    warmup_momentum=0.8,
    warmup_bias_lr=0.05,

    # LR Scheduler
    cos_lr=True,

    # -----------------------------------------------------
    # Dynamic Augmentation
    # Important because of oversampled potholes
    # -----------------------------------------------------
    mosaic=0.85,
    close_mosaic=5,         # Disable mosaic near the end
    mixup=0.15,

    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.4,

    degrees=15.0,
    translate=0.15,
    scale=0.6,
    shear=4.0,
    perspective=0.0003,

    # Spatial logic
    flipud=0.0,             # Roads should never be upside down
    fliplr=0.5,             # Horizontal flip is safe

    # -----------------------------------------------------
    # Loss Tuning
    # -----------------------------------------------------
    box=7.5,
    cls=0.7,                # Better balance than 1.0
    dfl=1.5,
    label_smoothing=0.02,

    # -----------------------------------------------------
    # Stability / Safety
    # -----------------------------------------------------
    patience=20,
    save=True,
    save_period=5,
    val=True,

    # -----------------------------------------------------
    # Hardware
    # -----------------------------------------------------
    amp=True,
    cache=False,
    device=0
)

# =========================================================
# FINAL REPORT
# =========================================================

print("\n" + "=" * 60)
print("🎉 PILOT RUN COMPLETE - JapanV2")
print("=" * 60)

print(f"mAP50      : {results.results_dict.get('metrics/mAP50(B)', 'N/A')}")
print(f"mAP50-95   : {results.results_dict.get('metrics/mAP50-95(B)', 'N/A')}")

print("\n📁 All results saved in:")
print(f"{project_path}/pilot_diagnostic_4to1/")

print("\n🔍 Next Step:")
print("Check confusion_matrix.png + pothole recall carefully")
print("If pothole recall improves → proceed to full 80 epochs")

print("=" * 60)

🚀 Starting JapanV2 PILOT RUN (20 Epochs)...
📦 Saving all weights/results directly to Google Drive...
🎯 Strategy: Fine-tuning on balanced 4:1 dataset
🧠 Focus: Improve pothole recall before full training
WARNING ⚠️ 'label_smoothing' is deprecated and will be removed in the future.
Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=6, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=5, cls=0.7, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/data.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.5, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, ko

In [ ]:
import os

# مسارات الفولدرات
train_images = set(os.listdir('/content/dataset_hybrid_4to1/dataset_hybrid_4to1/images'))
valid_images = set(os.listdir('/content/valid/valid/images'))

# البحث عن الصور المشتركة (التقاطع)
leakage = train_images.intersection(valid_images)

print("🔍 Checking for Data Leakage...")
if len(leakage) == 0:
    print("✅ SAFE! No overlapping images. Data Leakage = 0%")
else:
    print(f"❌ DANGER! Found {len(leakage)} leaked images!")
    print(list(leakage)[:5]) # عرض أول 5 صور مسربة

FileNotFoundError: [Errno 2] No such file or directory: '/content/dataset_hybrid_4to1/dataset_hybrid_4to1/images'